In [ ]:
import shutil

from extract_python.benches import DATA_PATH, TEST_DATA_PATH
from extract_python.benches.compare import (
    compare,
)
from extract_python.objects import InputDoc, OutputFormat
from extract_python.pipelines import DoclingPipeline, MarkerPipeline

In [ ]:
pdfs = [TEST_DATA_PATH / "computer_generated.pdf", TEST_DATA_PATH / "scanned.pdf"]
work_dir = DATA_PATH / "workdir"
comparison_dir = work_dir / "comparison"
docs = [InputDoc.from_path(pdf_path) for pdf_path in pdfs]

In [ ]:
from extract_python.pipelines import Pipeline

pipelines: list[Pipeline] = [DoclingPipeline(), MarkerPipeline()]

In [ ]:
from extract_python.objects import Result

if work_dir.exists():
    shutil.rmtree(work_dir)

for p in pipelines:
    pipeline_dir = p.registered_name.lower().replace("pipeline", "")
    output_path = work_dir / pipeline_dir
    output_path.mkdir(parents=True, exist_ok=True)
    mds: list[Result] = [
        r
        async for r in p.extract_content(  # noqa: PLE1142
            docs, output_format=OutputFormat.MARKDOWN, output_path=output_path
        )
    ]
    for md in mds:
        pages_path = output_path / md.output.path / "artifacts" / "pages.json"
        pages_path.write_text(md.output.pages.model_dump_json())

In [ ]:
from extract_python.benches.compare import discover_comparison

if comparison_dir.exists():
    shutil.rmtree(comparison_dir)

compare(discover_comparison(pdfs, work_dir), root=work_dir, output_path=comparison_dir)